# Task 7 Part A: Congruent vs Incongruent Speech Emotion Recognition (EMIS)

**Goal:** Test whether frozen SSL/supervised speech encoders exhibit text-bias by evaluating layer-wise probes on emotionally incongruent speech (EMIS dataset).

**Hypothesis:** If the midterm finding — that deep wav2vec 2.0 layers drift toward linguistic content while middle HuBERT layers retain prosodic information — is correct, then:
- Middle HuBERT layers should predict the **prosodic (audio) emotion** correctly on incongruent samples.
- Deep wav2vec 2.0 layers should be fooled and predict the **semantic (text) emotion**.
- The gap between these two accuracies at each layer = **text-bias metric**.

**Pipeline:**
1. Parse EMIS filenames to extract dual labels (text_emotion, audio_emotion) and metadata (sentence_id, ESD speaker, TTS system, explicit/implicit).
2. Extract hidden states from all 4 frozen models using the **same large-model configuration** as the main paper.
3. Train probe on **congruent samples only** (with sentence-level stratification to prevent leakage).
4. Test on **incongruent samples**; report target accuracy (prosodic) vs proxy accuracy (semantic) per layer.
5. Additionally split incongruent test set by **explicit vs implicit** text-emotion for mechanistic insight.

**Models (matching current main-paper configuration):**
- HuBERT: `facebook/hubert-large-ll60k` (25 layers, 1024 dim)
- wav2vec2: `facebook/wav2vec2-large-960h` (25 layers, 1024 dim)
- Whisper: `openai/whisper-medium` (25 layers, 1024 dim)
- WavLM: `microsoft/wavlm-large` (25 layers, 1024 dim)

**Data assumption:** EMIS WAVs sit in `/content/EMIS_audio/` with filenames following the schema `{sentence-ID}_{text-emotion}_{audio-emotion}_{ESD-speaker-ID}_{TTS-used}.wav`.

## 0. Setup

In [ ]:
!nvidia-smi

!pip install -q torch torchaudio transformers librosa soundfile
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q wandb

In [ ]:
# W&B login — use Colab Secrets (key icon in sidebar)
import os
import wandb

try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass

wandb.login()
print('W&B logged in')

In [ ]:
# ============================================================
# CONFIGURATION — matches main-paper large-model setup
# ============================================================

AUDIO_DIR = '/content/EMIS_audio'              # where EMIS WAVs live
FEATURE_DIR = '/content/emis_features'         # where to cache hidden states
RESULTS_DIR = '/content/results_task7'         # where to save CSVs and plots

os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS = {
    'HuBERT':   'facebook/hubert-large-ll60k',
    'wav2vec2': 'facebook/wav2vec2-large-960h',
    'Whisper':  'openai/whisper-medium',
    'WavLM':    'microsoft/wavlm-large',
}

# Expected: all four models produce 25 layers × 1024 dim on EMIS

WANDB_ENTITY = 'AGSER'
WANDB_PROJECT = 'linguistic-agnostic-ser'
WANDB_GROUP = 'task7_incongruent_EMIS'

print(f'Audio dir: {AUDIO_DIR}')
print(f'Feature cache: {FEATURE_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Models: {list(MODELS.keys())}')

## 1. Verify EMIS Dataset Integrity

In [ ]:
# ============================================================
# EMIS FILENAME PARSER & SANITY CHECK
# ============================================================
# Schema: {sentence-ID}_{text-emotion}_{audio-emotion}_{ESD-speaker-ID}_{TTS-used}.wav
# Example: 042_angry_happy_0011_XTTS.wav

import os
import pandas as pd
from collections import Counter

EXPECTED_EMOTIONS = {'angry', 'happy', 'neutral', 'sad'}

def parse_emis_filename(filename):
    """Parse an EMIS filename into its components.

    Returns a dict with sentence_id, text_emotion, audio_emotion, esd_speaker,
    tts_system, and a derived 'congruent' flag. Returns None if the filename
    doesn't match the expected 5-field schema or if emotion labels are invalid.
    """
    fname = os.path.splitext(os.path.basename(str(filename)))[0]
    parts = fname.split('_')
    if len(parts) != 5:
        return None
    sentence_id, text_emo, audio_emo, esd_spk, tts = parts
    if text_emo.lower() not in EXPECTED_EMOTIONS or audio_emo.lower() not in EXPECTED_EMOTIONS:
        return None
    return {
        'sentence_id': sentence_id,
        'text_emotion': text_emo.lower(),
        'audio_emotion': audio_emo.lower(),
        'esd_speaker': esd_spk,
        'tts_system': tts,
        'congruent': text_emo.lower() == audio_emo.lower(),
        'filename': os.path.basename(str(filename)),
    }

# Enumerate all EMIS files and validate
if not os.path.isdir(AUDIO_DIR):
    raise FileNotFoundError(f'EMIS directory not found: {AUDIO_DIR}. '
                            f'Download EMIS from IEEE DataPort first '
                            f'(https://ieee-dataport.org/documents/emotionally-incongruent-synthetic-speech-dataset-emis).')

all_wavs = sorted([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav')])
parsed = [parse_emis_filename(f) for f in all_wavs]
valid = [p for p in parsed if p is not None]
invalid_count = sum(1 for p in parsed if p is None)

print(f'Total WAVs found: {len(all_wavs)}')
print(f'Successfully parsed: {len(valid)}')
print(f'Failed to parse: {invalid_count}')

if invalid_count > 0:
    print('\nFirst few unparseable filenames:')
    for p, f in zip(parsed, all_wavs):
        if p is None:
            print(f'  {f}')
            if sum(1 for _ in (p for p in parsed[:20] if p is None)) > 5:
                break

assert len(valid) > 0, 'No valid EMIS samples found — check AUDIO_DIR and filename format.'

meta_df = pd.DataFrame(valid)
print('\nCongruent / Incongruent breakdown:')
print(meta_df['congruent'].value_counts())
print('\nTTS systems:', sorted(meta_df['tts_system'].unique()))
print('ESD speakers:', sorted(meta_df['esd_speaker'].unique()))
print(f'Unique sentences: {meta_df["sentence_id"].nunique()}')

# Save metadata for downstream analysis
meta_df.to_csv(os.path.join(RESULTS_DIR, 'emis_metadata.csv'), index=False)

## 2. Load Explicit vs Implicit Sentence Categorization

The EMIS abstract distinguishes **explicit** sentences (emotion word appears in text, e.g., "I am furious") from **implicit** sentences (emotion conveyed contextually, e.g., "I can't believe you did this again"). If the EMIS release ships a `sentences.csv` that labels this, we use it. Otherwise we treat all as one group and note the limitation.

**Expected location:** a metadata CSV in `AUDIO_DIR` or a sibling directory, with columns like `sentence_id`, `text`, `category` (or `type`), and `emotion`.

In [ ]:
# Try to locate the EMIS sentences metadata file
import glob

candidate_patterns = [
    os.path.join(AUDIO_DIR, '*.csv'),
    os.path.join(os.path.dirname(AUDIO_DIR), '*.csv'),
    '/content/*.csv',
]
candidates = []
for pat in candidate_patterns:
    candidates.extend(glob.glob(pat))

sentences_csv = None
for c in candidates:
    try:
        df = pd.read_csv(c)
        cols_lower = [col.lower() for col in df.columns]
        if any('explicit' in str(v).lower() or 'implicit' in str(v).lower() for v in df.values.flatten()[:50]):
            sentences_csv = c
            break
        if 'category' in cols_lower or 'type' in cols_lower:
            sentences_csv = c
            break
    except Exception:
        continue

explicit_map = {}  # sentence_id -> 'explicit' or 'implicit'

if sentences_csv:
    print(f'Found sentences metadata: {sentences_csv}')
    sent_df = pd.read_csv(sentences_csv)
    print(sent_df.head())
    # Try to build the mapping robustly
    id_col = next((c for c in sent_df.columns if 'id' in c.lower()), None)
    cat_col = next((c for c in sent_df.columns if c.lower() in ('category', 'type', 'explicit')), None)
    if id_col and cat_col:
        for _, row in sent_df.iterrows():
            sid = str(row[id_col])
            cat = str(row[cat_col]).lower().strip()
            if 'explicit' in cat:
                explicit_map[sid] = 'explicit'
            elif 'implicit' in cat:
                explicit_map[sid] = 'implicit'
        print(f'Mapped {len(explicit_map)} sentence IDs to explicit/implicit.')
else:
    print('No sentences metadata CSV found. Explicit/implicit split will be unavailable.')
    print('Results will still report target vs proxy accuracy on all incongruent samples.')

# Attach explicit/implicit label to metadata
meta_df['category'] = meta_df['sentence_id'].map(explicit_map).fillna('unknown')
print('\nCategory distribution:')
print(meta_df['category'].value_counts())

## 3. Feature Extraction with Frozen Encoders

In [ ]:
import torch
import numpy as np
import torchaudio
import librosa
from tqdm import tqdm
from transformers import AutoModel, AutoConfig, AutoFeatureExtractor


class FeatureExtractor:
    """Wraps a HuggingFace speech encoder with frozen weights and output_hidden_states=True.

    For Whisper, uses the encoder only (no decoder). For all others, uses the
    full model with raw waveform input.
    """

    def __init__(self, model_name, device='cuda'):
        self.device = device
        self.model_name = model_name
        self.is_whisper = 'whisper' in model_name.lower()

        config = AutoConfig.from_pretrained(model_name)
        config.output_hidden_states = True

        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.model.eval().to(device)
        for p in self.model.parameters():
            p.requires_grad = False

    def extract(self, waveform, sr=16000):
        """Mean-pool hidden states over time for every layer. Returns (num_layers, hidden_dim) numpy array."""
        target_sr = getattr(self.feature_extractor, 'sampling_rate', sr)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)

        inputs = self.feature_extractor(
            waveform.numpy(),
            sampling_rate=target_sr,
            return_tensors='pt',
        )

        if self.is_whisper and 'input_features' in inputs:
            model_inputs = {'input_features': inputs['input_features'].to(self.device)}
        else:
            model_inputs = {'input_values': inputs['input_values'].to(self.device)}

        with torch.no_grad():
            if self.is_whisper:
                outputs = self.model.encoder(**model_inputs)
            else:
                outputs = self.model(**model_inputs)

        pooled = [torch.mean(state, dim=1).squeeze(0).cpu().numpy() for state in outputs.hidden_states]
        stacked = np.stack(pooled, axis=0)  # (num_layers, hidden_dim)

        del outputs
        torch.cuda.empty_cache()
        return stacked

print('FeatureExtractor ready.')

In [ ]:
# ============================================================
# EXTRACT FEATURES FOR ALL MODELS (skip if cached)
# ============================================================

EXPECTED_LAYERS = 25
EXPECTED_DIM = 1024

# Use the validated metadata order to guarantee consistent indexing
ordered_filenames = meta_df['filename'].tolist()

for model_name, model_path in MODELS.items():
    hs_path = os.path.join(FEATURE_DIR, f'hidden_states_{model_name}_EMIS.npy')
    fn_path = os.path.join(FEATURE_DIR, f'filenames_{model_name}_EMIS.npy')

    if os.path.exists(hs_path) and os.path.exists(fn_path):
        cached = np.load(hs_path)
        print(f'[{model_name}] cached: {cached.shape} — skipping extraction')
        continue

    print(f'\n=== {model_name}: {model_path} ===')
    extractor = FeatureExtractor(model_path, device='cuda' if torch.cuda.is_available() else 'cpu')

    all_hidden = []
    kept_filenames = []
    failures = []

    for fname in tqdm(ordered_filenames, desc=model_name):
        audio_path = os.path.join(AUDIO_DIR, fname)
        try:
            wav, _ = librosa.load(audio_path, sr=16000, mono=True)
            wav_tensor = torch.from_numpy(wav).float()
            if wav_tensor.dim() == 1:
                wav_tensor = wav_tensor.unsqueeze(0)
            stacked = extractor.extract(wav_tensor, sr=16000)
            all_hidden.append(stacked)
            kept_filenames.append(fname)
        except Exception as e:
            failures.append((fname, str(e)))

    if failures:
        print(f'  WARNING: {len(failures)} files failed. First 3: {failures[:3]}')

    hidden_states = np.stack(all_hidden, axis=0)  # (n_samples, n_layers, hidden_dim)
    print(f'  shape: {hidden_states.shape}')

    # Soft sanity check — warn but don't abort if dims differ (e.g. if someone
    # swaps a model variant later)
    if hidden_states.shape[1] != EXPECTED_LAYERS or hidden_states.shape[2] != EXPECTED_DIM:
        print(f'  NOTE: expected ({EXPECTED_LAYERS}, {EXPECTED_DIM}) per sample; '
              f'got ({hidden_states.shape[1]}, {hidden_states.shape[2]}). '
              f'Verify model variant is correct.')

    np.save(hs_path, hidden_states)
    np.save(fn_path, np.array(kept_filenames, dtype=object))
    print(f'  saved: {hs_path}')

    del extractor
    torch.cuda.empty_cache()

print('\nAll feature extraction complete.')

## 4. Congruent-Train / Incongruent-Test Probing

For each layer independently:

1. Train logistic regression on **congruent** samples only, using `audio_emotion` as the label (which equals `text_emotion` for these samples).
2. Test on **incongruent** samples. Report:
   - **Target accuracy:** probe prediction matches `audio_emotion` (probe used prosody correctly).
   - **Proxy accuracy:** probe prediction matches `text_emotion` (probe was fooled by text).
   - **Text-bias gap:** `proxy_accuracy - target_accuracy` (positive means text-biased).

**Sentence-level stratification:** sentences appearing in the congruent train set are *excluded* from the incongruent test set. Without this, the probe could memorize sentence-specific artifacts.

**Multi-fold evaluation:** we use k-fold over unique sentence IDs to get stable estimates with standard deviations.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score

N_FOLDS = 5  # folds over unique sentence IDs
RANDOM_STATE = 42


def run_probing_single_model(hidden_states, filenames, meta_df, model_name,
                             restrict_category=None):
    """Run layer-wise congruent-train / incongruent-test probing.

    Parameters
    ----------
    hidden_states : (n_samples, n_layers, hidden_dim) array
    filenames     : list of filenames matching the rows of hidden_states
    meta_df       : full metadata DataFrame (indexed by filename column)
    model_name    : str, for logging
    restrict_category : None | 'explicit' | 'implicit'
        If set, the *incongruent test set* is restricted to that category.
        The congruent train set is always all congruent samples (more data is better).

    Returns
    -------
    pd.DataFrame with per-layer metrics averaged over folds.
    """
    # Map filename -> row index
    fn_to_idx = {fn: i for i, fn in enumerate(filenames)}

    # Align metadata to feature order (drop any rows without features)
    meta_aligned = meta_df[meta_df['filename'].isin(fn_to_idx)].copy()
    meta_aligned['row_idx'] = meta_aligned['filename'].map(fn_to_idx)
    meta_aligned = meta_aligned.sort_values('row_idx').reset_index(drop=True)

    n_layers = hidden_states.shape[1]
    all_emotions = sorted(set(meta_aligned['audio_emotion']).union(set(meta_aligned['text_emotion'])))
    le = LabelEncoder().fit(all_emotions)
    n_classes = len(all_emotions)
    chance = 1.0 / n_classes

    # Split rows into congruent / incongruent (+ category restriction on test)
    congruent_mask = meta_aligned['congruent'].values
    incongruent_mask = ~congruent_mask
    if restrict_category is not None:
        incongruent_mask = incongruent_mask & (meta_aligned['category'].values == restrict_category)

    cong_idx = np.where(congruent_mask)[0]
    incong_idx = np.where(incongruent_mask)[0]
    cong_sentences = meta_aligned.loc[cong_idx, 'sentence_id'].values
    incong_sentences = meta_aligned.loc[incong_idx, 'sentence_id'].values
    unique_sentences = np.array(sorted(set(cong_sentences).union(set(incong_sentences))))

    tag = f'{model_name}' + (f' [{restrict_category}]' if restrict_category else '')
    print(f'\n[{tag}] congruent train pool: {len(cong_idx)}, incongruent test pool: {len(incong_idx)}, '
          f'unique sentences: {len(unique_sentences)}, classes: {n_classes} (chance={chance:.3f})')

    if len(incong_idx) == 0:
        print(f'  [{tag}] empty test set — skipping.')
        return None

    # K-fold over unique sentence IDs ensures no sentence leakage
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    sent_splits = list(kf.split(unique_sentences))

    fold_records = []  # per-fold, per-layer metrics

    for fold_i, (train_sent_i, test_sent_i) in enumerate(sent_splits):
        train_sentences = set(unique_sentences[train_sent_i])
        test_sentences = set(unique_sentences[test_sent_i])

        train_rows = cong_idx[np.isin(cong_sentences, list(train_sentences))]
        test_rows = incong_idx[np.isin(incong_sentences, list(test_sentences))]

        if len(train_rows) < 2 * n_classes or len(test_rows) == 0:
            continue

        y_train = le.transform(meta_aligned.loc[train_rows, 'audio_emotion'].values)
        y_target = le.transform(meta_aligned.loc[test_rows, 'audio_emotion'].values)
        y_proxy = le.transform(meta_aligned.loc[test_rows, 'text_emotion'].values)

        if len(np.unique(y_train)) < 2:
            continue

        for layer_idx in range(n_layers):
            X_train = hidden_states[train_rows, layer_idx, :]
            X_test = hidden_states[test_rows, layer_idx, :]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = LogisticRegression(max_iter=2000, solver='lbfgs', C=1.0, n_jobs=1)
            clf.fit(X_train_s, y_train)
            preds = clf.predict(X_test_s)

            target_acc = accuracy_score(y_target, preds)
            proxy_acc = accuracy_score(y_proxy, preds)
            target_f1 = f1_score(y_target, preds, average='weighted', zero_division=0)

            fold_records.append({
                'fold': fold_i,
                'layer': layer_idx,
                'target_acc': target_acc,
                'proxy_acc': proxy_acc,
                'target_f1': target_f1,
                'text_bias': proxy_acc - target_acc,
                'n_train': len(train_rows),
                'n_test': len(test_rows),
            })

    if not fold_records:
        print(f'  [{tag}] no usable folds — returning None.')
        return None

    fold_df = pd.DataFrame(fold_records)
    summary = fold_df.groupby('layer').agg(
        target_acc_mean=('target_acc', 'mean'),
        target_acc_std=('target_acc', 'std'),
        proxy_acc_mean=('proxy_acc', 'mean'),
        proxy_acc_std=('proxy_acc', 'std'),
        target_f1_mean=('target_f1', 'mean'),
        text_bias_mean=('text_bias', 'mean'),
        text_bias_std=('text_bias', 'std'),
        n_train=('n_train', 'mean'),
        n_test=('n_test', 'mean'),
    ).reset_index()
    summary['model'] = model_name
    summary['category'] = restrict_category or 'all'
    summary['chance'] = chance

    # Report
    print(f'  {"Layer":>5} | {"Target":>7} (std) | {"Proxy":>7} (std) | {"Bias":>6}')
    for _, row in summary.iterrows():
        layer_name = 'CNN' if row['layer'] == 0 else f'L{int(row["layer"])}'
        print(f'  {layer_name:>5} | {row["target_acc_mean"]:.3f} ({row["target_acc_std"]:.3f}) | '
              f'{row["proxy_acc_mean"]:.3f} ({row["proxy_acc_std"]:.3f}) | '
              f'{row["text_bias_mean"]:+.3f}')

    return summary

print('Probing function ready.')

In [ ]:
# ============================================================
# RUN PROBING — all models × {all, explicit, implicit}
# ============================================================

all_results = []
have_category = (meta_df['category'] != 'unknown').any()
categories_to_run = [None] + (['explicit', 'implicit'] if have_category else [])

for model_name in MODELS.keys():
    hs_path = os.path.join(FEATURE_DIR, f'hidden_states_{model_name}_EMIS.npy')
    fn_path = os.path.join(FEATURE_DIR, f'filenames_{model_name}_EMIS.npy')

    if not (os.path.exists(hs_path) and os.path.exists(fn_path)):
        print(f'[{model_name}] features missing — run feature extraction first.')
        continue

    hidden_states = np.load(hs_path)
    filenames = np.load(fn_path, allow_pickle=True).tolist()

    print(f'\n{"="*60}')
    print(f'  {model_name} — shape: {hidden_states.shape}')
    print(f'{"="*60}')

    for cat in categories_to_run:
        summary = run_probing_single_model(
            hidden_states, filenames, meta_df,
            model_name=model_name,
            restrict_category=cat,
        )
        if summary is not None:
            all_results.append(summary)
            tag = cat or 'all'
            csv_path = os.path.join(RESULTS_DIR, f'probing_{model_name}_{tag}.csv')
            summary.to_csv(csv_path, index=False)
            print(f'  saved: {csv_path}')

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(os.path.join(RESULTS_DIR, 'probing_all_combined.csv'), index=False)
    print(f'\nCombined results: {combined.shape} rows')
else:
    print('No probing results produced — check feature extraction and metadata.')

## 5. Plot: Layer-Wise Target vs Proxy Accuracy

This is the key figure of Task 7. One panel per model, two curves (target and proxy accuracy), shaded region = text-bias gap, dotted line at chance.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
})


def plot_text_bias(combined_df, category='all', save_name=None):
    """One panel per model, target vs proxy accuracy per layer."""
    sub = combined_df[combined_df['category'] == category]
    models_present = [m for m in MODELS.keys() if m in sub['model'].unique()]
    if not models_present:
        print(f'No results for category={category}')
        return None

    n = len(models_present)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, m in zip(axes, models_present):
        d = sub[sub['model'] == m].sort_values('layer')
        layers = d['layer'].values
        chance = d['chance'].iloc[0]

        ax.plot(layers, d['target_acc_mean'], 'o-', color='#2196F3',
                linewidth=2, markersize=4, label='Target (audio emotion)', zorder=3)
        ax.fill_between(layers,
                        d['target_acc_mean'] - d['target_acc_std'],
                        d['target_acc_mean'] + d['target_acc_std'],
                        color='#2196F3', alpha=0.15)

        ax.plot(layers, d['proxy_acc_mean'], 's--', color='#E53935',
                linewidth=2, markersize=4, label='Proxy (text emotion)', zorder=3)
        ax.fill_between(layers,
                        d['proxy_acc_mean'] - d['proxy_acc_std'],
                        d['proxy_acc_mean'] + d['proxy_acc_std'],
                        color='#E53935', alpha=0.15)

        # Text-bias shading — only where proxy > target
        ax.fill_between(layers,
                        d['target_acc_mean'],
                        d['proxy_acc_mean'],
                        where=d['proxy_acc_mean'] > d['target_acc_mean'],
                        color='#FFB74D', alpha=0.25, label='Text-bias region')

        ax.axhline(chance, color='gray', linestyle=':', alpha=0.6,
                   label=f'Chance ({chance:.2f})')

        ax.set_xlabel('Layer')
        ax.set_title(m, fontweight='bold')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', framealpha=0.9)
        ax.set_ylim(0, 1.0)

    axes[0].set_ylabel('Accuracy on Incongruent Samples')
    cat_title = {'all': 'all incongruent', 'explicit': 'explicit text only', 'implicit': 'implicit text only'}[category]
    fig.suptitle(f'Layer-Wise Text-Bias on EMIS ({cat_title})\n'
                 f'Train on congruent (sentence-stratified) -> Test on incongruent',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()

    if save_name:
        out_path = os.path.join(RESULTS_DIR, save_name)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'saved: {out_path}')
    plt.show()
    return fig


if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    plot_text_bias(combined, category='all', save_name='text_bias_all.png')
    if have_category:
        plot_text_bias(combined, category='explicit', save_name='text_bias_explicit.png')
        plot_text_bias(combined, category='implicit', save_name='text_bias_implicit.png')

## 6. Summary Table: Best & Worst Layers per Model

In [ ]:
if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    all_view = combined[combined['category'] == 'all']

    summary_rows = []
    for m in MODELS.keys():
        d = all_view[all_view['model'] == m]
        if d.empty:
            continue
        best_target = d.loc[d['target_acc_mean'].idxmax()]
        worst_target = d.loc[d['target_acc_mean'].idxmin()]
        max_bias = d.loc[d['text_bias_mean'].idxmax()]
        summary_rows.append({
            'model': m,
            'best_target_layer': int(best_target['layer']),
            'best_target_acc': round(best_target['target_acc_mean'], 4),
            'worst_target_layer': int(worst_target['layer']),
            'worst_target_acc': round(worst_target['target_acc_mean'], 4),
            'max_bias_layer': int(max_bias['layer']),
            'max_bias_value': round(max_bias['text_bias_mean'], 4),
            'chance': round(d['chance'].iloc[0], 4),
        })

    summary_tbl = pd.DataFrame(summary_rows)
    print(summary_tbl.to_string(index=False))
    summary_tbl.to_csv(os.path.join(RESULTS_DIR, 'best_worst_summary.csv'), index=False)
    print(f'\nsaved: {os.path.join(RESULTS_DIR, "best_worst_summary.csv")}')

## 7. Log Results to W&B

In [ ]:
if all_results:
    combined = pd.concat(all_results, ignore_index=True)

    for m in MODELS.keys():
        for cat in combined[combined['model'] == m]['category'].unique():
            d = combined[(combined['model'] == m) & (combined['category'] == cat)].sort_values('layer')
            if d.empty:
                continue

            run = wandb.init(
                entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                group=WANDB_GROUP,
                name=f'EMIS_{m}_{cat}',
                reinit=True,
                config={
                    'model': m,
                    'model_path': MODELS[m],
                    'dataset': 'EMIS',
                    'category': cat,
                    'n_folds': N_FOLDS,
                    'task': 'task7_incongruent',
                },
            )

            for _, row in d.iterrows():
                wandb.log({
                    'layer': int(row['layer']),
                    'target_acc_mean': row['target_acc_mean'],
                    'target_acc_std': row['target_acc_std'],
                    'proxy_acc_mean': row['proxy_acc_mean'],
                    'proxy_acc_std': row['proxy_acc_std'],
                    'text_bias_mean': row['text_bias_mean'],
                    'target_f1_mean': row['target_f1_mean'],
                })

            wandb.summary['best_target_layer'] = int(d.loc[d['target_acc_mean'].idxmax(), 'layer'])
            wandb.summary['best_target_acc'] = float(d['target_acc_mean'].max())
            wandb.summary['max_text_bias'] = float(d['text_bias_mean'].max())
            wandb.summary['max_text_bias_layer'] = int(d.loc[d['text_bias_mean'].idxmax(), 'layer'])

            run.finish()

    print('Logged all runs to W&B.')
else:
    print('Nothing to log.')

## 8. Package Outputs for Discovery Transfer

In [ ]:
# Zip features + results for transfer back to Discovery
!cd /content && zip -rq task7_emis_bundle.zip emis_features/ results_task7/
print('Bundle:', os.path.getsize('/content/task7_emis_bundle.zip') / 1e6, 'MB')

# If running in Colab, download
try:
    from google.colab import files
    files.download('/content/task7_emis_bundle.zip')
except Exception:
    print('Not in Colab — transfer manually:')
    print('  scp task7_emis_bundle.zip minooahm@discovery1.usc.edu:/scratch1/minooahm/ser_data/')